In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyhdf.SD import SD, SDC
import xarray as xr

from pyhdf.HDF import HDF, HC
from pyhdf.VS import VS

In [2]:
acp_file = '../data/MISR/MISR_AM1_ACP_MIXTURE_F06_0022.hdf'
sd = SD(acp_file, SDC.READ)
hdf = HDF(acp_file, HC.READ)
vs = VS(hdf)

In [3]:
# Example code for "Mixture Data" formatting
# mixture_number = 31
# categories = [vs.attach('Mixture Data').fieldinfo()[i][0] for i in range(10)]
# values = [vs.attach('Mixture Data')[mixture_number-1][i] for i in range(10)]
# for i, (c, v) in enumerate(zip(categories, values)):
#     print('{}. {:s}: {}'.format(i, c, v))

In [4]:
mixture_data = vs.attach('Mixture Data')[:]
ssalb = np.zeros((4, len(mixture_data)))
angstrom = np.zeros(len(mixture_data))
for i in range(len(mixture_data)):
    ssalb[:, i] = mixture_data[i][5]
    angstrom[i] = mixture_data[i][9]

In [5]:
scatter_angle = sd.select('Scattering angle')[:]
phase = sd.select('Mixture Spectral Phase Functions')[:]
band = sd.select('Band')[:]
acp = xr.Dataset(
    data_vars=dict(
        phase=(['band', 'angle', 'mixture'], phase),
        ssalb=(['band', 'mixture'], ssalb),
        angstrom=(['mixture'], angstrom)
    ),
    coords=dict(
        band=band,
        angle=scatter_angle,
        mixture=np.arange(1,75)
    )
)
acp = acp.assign_coords(theta = np.radians(acp.angle))
acp['g_integrand'] = acp.phase * np.cos(acp.theta) * np.sin(acp.theta)
acp['g'] = 0.5 * acp.g_integrand.integrate('theta')
acp = acp.drop_vars('g_integrand')

acp['angle'] = acp.angle.assign_attrs(long_name='Scattering angle', units='degrees')
acp['band'] = acp.band.assign_attrs(long_name='Wavelength', units='um')
acp['theta'] = acp.theta.assign_attrs(long_name='Scattering angle', units='radians')

acp.to_netcdf('../data/MISR/MISR_AM1_ACP_MIXTURE_F06_0022.nc')